# Laboratorio 09: Interpretabilidad de Modelos con SHAP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gonzalezulises/formacion-docente-bi-faces/blob/main/modulo-04-predictivo-proyecto/laboratorios/lab_09_interpretabilidad_ejercicios.ipynb)

## Objetivos de Aprendizaje

Al finalizar este laboratorio, usted será capaz de:

1. **Comprender** por qué la interpretabilidad es crucial en modelos predictivos
2. **Aplicar** SHAP (SHapley Additive exPlanations) para explicar predicciones
3. **Interpretar** valores SHAP a nivel individual y global
4. **Comunicar** hallazgos de modelos a audiencias no técnicas

## ¿Por qué Interpretabilidad?

> *"Un modelo que no puedes explicar es un modelo en el que no puedes confiar."*

En contextos universitarios y de gestión, no basta con que un modelo prediga bien. Necesitamos:

- **Explicar** a stakeholders por qué el modelo hace ciertas predicciones
- **Validar** que el modelo aprende patrones sensatos (no sesgos)
- **Tomar decisiones** informadas basadas en los factores más importantes
- **Cumplir** con requisitos éticos y de transparencia

## Duración estimada: 90 minutos

---

## Parte 0: Configuración del Entorno

In [ ]:
# Instalar SHAP si no está disponible
!pip install shap -q

In [ ]:
# Importaciones necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

# Interpretabilidad
import shap

# Configuración
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)
np.random.seed(42)

print("✅ Librerías cargadas correctamente")
print(f"SHAP versión: {shap.__version__}")

## Parte 1: Cargar y Preparar Datos

Usaremos el dataset de **rendimiento académico** para predecir qué estudiantes están en **riesgo de reprobar**.

In [ ]:
# Cargar datos
url = "https://raw.githubusercontent.com/gonzalezulises/formacion-docente-bi-faces/main/datasets/universidad/rendimiento_academico.csv"
df = pd.read_csv(url)

print(f"Dataset: {df.shape[0]} estudiantes, {df.shape[1]} variables")
df.head()

In [ ]:
# Exploración rápida
df.info()

### Ejercicio 1.1: Crear Variable Objetivo

Cree una variable binaria `en_riesgo` que sea 1 si el promedio del estudiante es menor a 10 (reprobado), y 0 en caso contrario.

In [ ]:
# EJERCICIO 1.1: Crear variable objetivo
# Hint: Use np.where() o una comparación directa

# df['en_riesgo'] = ___

# Verificar distribución
# print(df['en_riesgo'].value_counts(normalize=True))

### Ejercicio 1.2: Preparar Variables Predictoras

Seleccione las variables predictoras relevantes y prepare los datos para modelado:
- Codifique variables categóricas
- Maneje valores faltantes si los hay
- Divida en train/test (80/20)

In [ ]:
# EJERCICIO 1.2: Preparar datos

# Paso 1: Seleccionar columnas predictoras (excluir ID, promedio, y la variable objetivo)
# predictores = ['asistencia', 'horas_estudio', 'trabaja', 'tiene_beca', ...]

# Paso 2: Codificar variables categóricas si es necesario
# le = LabelEncoder()
# df['carrera_encoded'] = le.fit_transform(df['carrera'])

# Paso 3: Crear X e y
# X = df[predictores]
# y = df['en_riesgo']

# Paso 4: Dividir en train/test
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

---

## Parte 2: Entrenar Modelo Base

Entrenaremos un modelo de **Random Forest** que luego interpretaremos con SHAP.

### Ejercicio 2.1: Entrenar Random Forest

In [ ]:
# EJERCICIO 2.1: Entrenar modelo

# modelo = RandomForestClassifier(
#     n_estimators=100,
#     max_depth=10,
#     random_state=42
# )
# modelo.fit(X_train, y_train)

# Evaluar
# y_pred = modelo.predict(X_test)
# print(classification_report(y_test, y_pred))

### Ejercicio 2.2: Feature Importance Tradicional

Antes de SHAP, veamos la importancia de variables tradicional de Random Forest.

**Limitación**: Esta importancia solo dice *qué tan frecuentemente* se usa una variable, no *cómo* afecta la predicción.

In [ ]:
# EJERCICIO 2.2: Visualizar feature importance tradicional

# importance_df = pd.DataFrame({
#     'variable': X.columns,
#     'importancia': modelo.feature_importances_
# }).sort_values('importancia', ascending=True)

# plt.figure(figsize=(10, 6))
# plt.barh(importance_df['variable'], importance_df['importancia'])
# plt.xlabel('Importancia')
# plt.title('Feature Importance - Random Forest')
# plt.tight_layout()
# plt.show()

---

## Parte 3: Introducción a SHAP

### ¿Qué es SHAP?

**SHAP** (SHapley Additive exPlanations) se basa en la **teoría de juegos cooperativos**. La idea:

1. Cada variable es un "jugador" que contribuye a la predicción
2. El valor SHAP de una variable es su **contribución marginal promedio** a la predicción
3. Los valores SHAP suman exactamente la diferencia entre la predicción y el valor base

### Ventajas sobre Feature Importance tradicional:

| Aspecto | Feature Importance | SHAP |
|---------|-------------------|------|
| Dirección del efecto | ❌ No indica | ✅ Positivo/Negativo |
| Explicación individual | ❌ Solo global | ✅ Por cada predicción |
| Consistencia teórica | ⚠️ Variable | ✅ Axiomática |
| Interacciones | ❌ No captura | ✅ Puede capturar |

### Ejercicio 3.1: Calcular Valores SHAP

In [ ]:
# EJERCICIO 3.1: Calcular valores SHAP

# Crear el explicador SHAP para el modelo
# Para Random Forest, usamos TreeExplainer (más eficiente)

# explainer = shap.TreeExplainer(modelo)

# Calcular valores SHAP para el conjunto de test
# shap_values = explainer.shap_values(X_test)

# Para clasificación binaria, shap_values es una lista de 2 elementos
# shap_values[0] = contribuciones para clase 0 (no en riesgo)
# shap_values[1] = contribuciones para clase 1 (en riesgo)

# print(f"Forma de shap_values: {len(shap_values)} clases")
# print(f"Valores para clase 1: {shap_values[1].shape}")

---

## Parte 4: Visualizaciones SHAP

SHAP ofrece varias visualizaciones poderosas. Exploraremos las más importantes.

### Ejercicio 4.1: Summary Plot (Importancia Global)

El **summary plot** muestra:
- **Eje Y**: Variables ordenadas por importancia
- **Eje X**: Valor SHAP (impacto en la predicción)
- **Color**: Valor de la variable (rojo = alto, azul = bajo)

In [ ]:
# EJERCICIO 4.1: Crear summary plot

# plt.figure(figsize=(10, 8))
# shap.summary_plot(shap_values[1], X_test, show=False)
# plt.title('SHAP Summary Plot - Factores de Riesgo Académico')
# plt.tight_layout()
# plt.show()

### Pregunta de Reflexión 4.1

Observe el summary plot y responda:

1. ¿Cuáles son las 3 variables más importantes para predecir riesgo académico?
2. Para la variable más importante, ¿valores altos aumentan o disminuyen el riesgo?
3. ¿Hay alguna relación que le sorprenda o que contradiga su intuición?

*Escriba sus respuestas aquí:*

```
1. 
2. 
3. 
```

### Ejercicio 4.2: Bar Plot (Importancia Absoluta)

Una vista más simple que muestra solo la importancia promedio (sin dirección).

In [ ]:
# EJERCICIO 4.2: Crear bar plot

# plt.figure(figsize=(10, 6))
# shap.summary_plot(shap_values[1], X_test, plot_type="bar", show=False)
# plt.title('Importancia de Variables (SHAP)')
# plt.tight_layout()
# plt.show()

### Ejercicio 4.3: Dependence Plot (Efecto de una Variable)

El **dependence plot** muestra cómo el valor de UNA variable afecta la predicción.

In [ ]:
# EJERCICIO 4.3: Crear dependence plot para la variable más importante

# Reemplace 'variable_importante' con el nombre de la variable más importante según el summary plot

# plt.figure(figsize=(10, 6))
# shap.dependence_plot(
#     'asistencia',  # Variable a analizar
#     shap_values[1],
#     X_test,
#     show=False
# )
# plt.title('Efecto de Asistencia en Riesgo Académico')
# plt.tight_layout()
# plt.show()

### Pregunta de Reflexión 4.3

Observe el dependence plot:

1. ¿La relación es lineal o hay puntos de inflexión?
2. ¿A partir de qué valor de la variable el efecto cambia significativamente?
3. ¿Cómo usaría esta información para una intervención?

*Escriba sus respuestas aquí:*

---

## Parte 5: Explicaciones Individuales

Una de las mayores fortalezas de SHAP es poder explicar **cada predicción individual**.

### Ejercicio 5.1: Waterfall Plot (Explicación de una Predicción)

In [ ]:
# EJERCICIO 5.1: Explicar una predicción individual

# Seleccionar un estudiante en riesgo del conjunto de test
# idx_riesgo = np.where(y_test == 1)[0][0]  # Primer estudiante en riesgo

# Crear objeto Explanation para visualización moderna
# shap_explanation = shap.Explanation(
#     values=shap_values[1][idx_riesgo],
#     base_values=explainer.expected_value[1],
#     data=X_test.iloc[idx_riesgo],
#     feature_names=X_test.columns.tolist()
# )

# plt.figure(figsize=(10, 6))
# shap.waterfall_plot(shap_explanation, show=False)
# plt.title(f'¿Por qué este estudiante está en riesgo?')
# plt.tight_layout()
# plt.show()

### Ejercicio 5.2: Force Plot (Vista Compacta)

In [ ]:
# EJERCICIO 5.2: Force plot para el mismo estudiante

# shap.initjs()  # Necesario para visualización interactiva

# shap.force_plot(
#     explainer.expected_value[1],
#     shap_values[1][idx_riesgo],
#     X_test.iloc[idx_riesgo],
#     matplotlib=True
# )

### Ejercicio 5.3: Comparar Dos Estudiantes

Compare un estudiante en riesgo con uno que no lo está.

In [ ]:
# EJERCICIO 5.3: Comparar dos estudiantes

# Encontrar un estudiante NO en riesgo
# idx_no_riesgo = np.where(y_test == 0)[0][0]

# Crear figura con dos subplots
# fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Para estudiante en riesgo
# ...

# Para estudiante no en riesgo
# ...

---

## Parte 6: Aplicación Práctica - Sistema de Alertas

Ahora usaremos SHAP para diseñar un **sistema de alertas tempranas** para estudiantes en riesgo.

### Ejercicio 6.1: Identificar Factores Modificables

No todos los factores de riesgo son modificables. Clasifique las variables:

In [ ]:
# EJERCICIO 6.1: Clasificar factores

# Revise las variables y clasifíquelas:

factores_modificables = [
    # Ejemplo: 'asistencia', 'horas_estudio'
    # Complete la lista...
]

factores_no_modificables = [
    # Ejemplo: 'edad', 'genero'
    # Complete la lista...
]

factores_parcialmente_modificables = [
    # Ejemplo: 'trabaja' (el estudiante puede decidir, pero hay restricciones)
    # Complete la lista...
]

### Ejercicio 6.2: Crear Reporte de Intervención

Para un estudiante en riesgo, genere un reporte que:
1. Identifique los factores que más contribuyen a su riesgo
2. Sugiera intervenciones específicas para factores modificables

In [ ]:
# EJERCICIO 6.2: Generar reporte de intervención

def generar_reporte_intervencion(idx_estudiante, X_test, shap_values, factores_modificables):
    """
    Genera un reporte de intervención para un estudiante en riesgo.
    
    Parámetros:
    - idx_estudiante: índice del estudiante en X_test
    - X_test: DataFrame con datos del estudiante
    - shap_values: valores SHAP calculados
    - factores_modificables: lista de variables que se pueden intervenir
    
    Retorna:
    - Diccionario con el reporte
    """
    # Obtener datos del estudiante
    estudiante = X_test.iloc[idx_estudiante]
    shap_estudiante = shap_values[1][idx_estudiante]
    
    # Crear DataFrame de contribuciones
    contribuciones = pd.DataFrame({
        'variable': X_test.columns,
        'valor': estudiante.values,
        'contribucion_riesgo': shap_estudiante
    }).sort_values('contribucion_riesgo', ascending=False)
    
    # Filtrar solo factores modificables que aumentan riesgo
    # intervenciones = contribuciones[
    #     (contribuciones['variable'].isin(factores_modificables)) & 
    #     (contribuciones['contribucion_riesgo'] > 0)
    # ]
    
    # TODO: Complete la función
    # 1. Identifique los top 3 factores modificables que aumentan riesgo
    # 2. Para cada uno, sugiera una intervención específica
    # 3. Retorne un diccionario con el reporte
    
    pass

# Probar la función
# reporte = generar_reporte_intervencion(idx_riesgo, X_test, shap_values, factores_modificables)
# print(reporte)

---

## Parte 7: Comunicación de Resultados

Los valores SHAP son técnicos. Necesitamos traducirlos a lenguaje que un decano o coordinador pueda entender.

### Ejercicio 7.1: Redactar Hallazgos

Basándose en su análisis SHAP, redacte 3 hallazgos principales en lenguaje no técnico.

**Ejemplo de hallazgo bien redactado:**

> "La asistencia a clases es el factor más importante para predecir el éxito académico. Estudiantes con menos del 70% de asistencia tienen 3 veces más probabilidad de reprobar. Recomendamos implementar un sistema de alertas cuando la asistencia baja del 80%."

**Sus hallazgos:**

1. *Escriba aquí...*

2. *Escriba aquí...*

3. *Escriba aquí...*

### Ejercicio 7.2: Limitaciones y Consideraciones Éticas

Todo modelo predictivo tiene limitaciones. Documente las de este análisis:

**Limitaciones técnicas:**

1. *Escriba aquí...*

2. *Escriba aquí...*

**Consideraciones éticas:**

1. ¿Qué pasa si el modelo tiene sesgos contra ciertos grupos?

2. ¿Cómo evitar que las predicciones se conviertan en profecías autocumplidas?

3. ¿Quién debe tener acceso a estas predicciones?

*Reflexione y escriba sus consideraciones aquí...*

---

## Parte 8: Extensión - SHAP con Otros Modelos

SHAP funciona con cualquier modelo. Compare la interpretación entre diferentes algoritmos.

### Ejercicio 8.1: Comparar con Regresión Logística

In [ ]:
# EJERCICIO 8.1: SHAP con Regresión Logística

# Escalar datos para regresión logística
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# Entrenar modelo
# modelo_log = LogisticRegression(random_state=42, max_iter=1000)
# modelo_log.fit(X_train_scaled, y_train)

# Para modelos lineales, usamos LinearExplainer
# explainer_log = shap.LinearExplainer(modelo_log, X_train_scaled)
# shap_values_log = explainer_log.shap_values(X_test_scaled)

# Comparar summary plots
# ...

### Pregunta de Reflexión Final

¿Las importancias de variables son consistentes entre Random Forest y Regresión Logística? ¿Qué podría explicar las diferencias?

---

## Resumen y Próximos Pasos

### Lo que aprendió:

1. **SHAP** proporciona explicaciones consistentes y aditivas para cualquier modelo
2. El **summary plot** muestra importancia global con dirección del efecto
3. El **waterfall plot** explica predicciones individuales
4. La interpretabilidad permite **intervenciones informadas**
5. Siempre considere **limitaciones éticas** de los modelos predictivos

### Para su proyecto integrador:

- Use SHAP para explicar su modelo predictivo
- Incluya al menos 2 visualizaciones SHAP
- Traduzca los hallazgos a lenguaje no técnico
- Discuta las implicaciones para intervención

### Recursos adicionales:

- [Documentación oficial SHAP](https://shap.readthedocs.io/)
- [Interpretable Machine Learning Book](https://christophm.github.io/interpretable-ml-book/) - Christoph Molnar
- [SHAP Paper original](https://arxiv.org/abs/1705.07874) - Lundberg & Lee (2017)

---

**Tiempo estimado de completar ejercicios:** 90 minutos

**Entregable:** Notebook completado con:
- Todos los ejercicios resueltos
- Preguntas de reflexión respondidas
- Al menos 3 visualizaciones SHAP
- Reporte de intervención para un estudiante